# miniLLM v2.2 — mettre à jour la fiche du modèle (sans renvoyer les poids)

Régénère `README.md`, `evals/resultats_automatiques.md` et `evals/test_connaissances.md` depuis le dépôt de code et les évaluations déjà faites, puis envoie **uniquement ces 3 fichiers** vers le dépôt Hugging Face existant. Les poids (260 Mo) ne sont pas renvoyés. Aucun GPU nécessaire, ~1 minute.

In [ ]:
import os, sys, glob, json, shutil
from google.colab import drive
drive.mount("/content/drive")
cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2") + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
EVAL_DIR = f"{BASE}/evals_v22"
CODE_DIR = "/content/minillm_v2"
!pip -q install huggingface_hub
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
import release_tools as rt
REPO_ID = "bonopassale/" + json.load(open(f"{EVAL_DIR}/fiche.json"))["repo_name"]
rel = f"{CODE_DIR}/release/v2.2"
OUT = "/content/maj_fiche"
os.makedirs(f"{OUT}/evals", exist_ok=True)
shutil.copy(f"{rel}/README.md", f"{OUT}/README.md")
open(f"{OUT}/evals/resultats_automatiques.md", "w", encoding="utf-8").write(rt.build_evals_md(EVAL_DIR, f"{rel}/evals_extra.md", "2.2"))
open(f"{OUT}/evals/test_connaissances.md", "w", encoding="utf-8").write(rt.knowledge_questions_md())
assert REPO_ID.split("/")[1] in open(f"{OUT}/README.md", encoding="utf-8").read()
print("Dépôt :", REPO_ID)
print({f: os.path.getsize(f"{OUT}/{f}") for f in ("README.md", "evals/resultats_automatiques.md", "evals/test_connaissances.md")})


In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    HF_TOKEN = getpass("Token Hugging Face (write) : ")
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_folder(folder_path=OUT, repo_id=REPO_ID, repo_type="model", commit_message="Mise à jour de la fiche : catégories du test de connaissances, mise en forme")
print("✓ Fiche mise à jour :", f"https://huggingface.co/{REPO_ID}")
